In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Controls — immediately after Drive mount; safe for Runtime → Run all
DRIVE_ROOT='/content/drive/MyDrive/OpenPlaque'
OUTPUT_ROOT=DRIVE_ROOT + '/Left_Coronary_Coordinate_Reconciliation_v1'
SOURCE_CACHE=DRIVE_ROOT + '/Cache/Secondary_3D_Vesselness_Topology_v1'
MAX_CANDIDATE_FILES=500


# OpenPlaque — Left-coronary coordinate / segmentation reconciliation

This diagnostic explains why the independent TotalSegmentator coronary masks agree almost perfectly with the RCA but were tens of millimeters from the LAD/secondary centerlines used by the first source-geometry integration.

The experiment is deliberately conservative:
- RCA is the positive-control source-coordinate anchor.
- Current and legacy TotalSegmentator masks are tested separately, plus union and intersection.
- Source-CCTA HU is sampled directly along every discovered LAD/secondary centerline candidate.
- Stored LPS coordinates are compared with LPS recomputed from any stored source voxel `(z,y,x)` coordinates.
- A specific RAS→LPS sign-flip hypothesis is audited, but no arbitrary free rigid registration or translation is allowed.
- Multiple OpenPlaque centerline files are discovered and ranked, so an outdated/wrong left-centerline file can be distinguished from TotalSegmentator under-segmentation.


In [ ]:
!pip -q install SimpleITK scipy pandas matplotlib pytest
import shutil, sys
from pathlib import Path
repo=Path('/content/OpenPlaque')
if repo.exists(): shutil.rmtree(repo)
!git clone -q --depth 1 --branch left-coronary-coordinate-reconciliation-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
sys.path.insert(0,'/content/OpenPlaque/src')


In [ ]:
# Fast unit checks before touching the Drive data
!cd /content/OpenPlaque && PYTHONPATH=/content/OpenPlaque/src pytest -q tests/test_left_coronary_reconciliation.py


In [ ]:
from openplaque.left_coronary_reconciliation import run
result = run(
    drive_root=DRIVE_ROOT,
    output_root=OUTPUT_ROOT,
    source_cache=SOURCE_CACHE,
    max_candidate_files=MAX_CANDIDATE_FILES,
)
print('PRIMARY CONCLUSION:', result['conclusion']['primary_conclusion'])
print('RCA POSITIVE CONTROL:', result['conclusion']['rca_positive_control'])
print('LAD:', result['conclusion']['lad_conclusion'])
print('SECONDARY:', result['conclusion']['secondary_conclusion'])
print('REPORT:', result['report'])
print('ZIP:', result['zip'])
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LEFT_CORONARY_COORDINATE_RECONCILIATION_REPORT_BACK.zip')
